# CatBoost any-wasting model with Optuna (30 trials)

This notebook models `any_wasting` using the same 23 predictors and complete-case workflow as the severe-wasting CatBoost notebook. It uses **Optuna** for hyperparameter tuning with a maximum of 30 trials, evaluates the selected model on a held-out test set, and then refits the selected configuration on all eligible complete cases.

**Outcome:** `any_wasting`. **Predictors:** the same 23 variables as the companion severe-wasting model.

**Important:** full-data predictions from the final refitted model are in-sample. Use held-out test metrics for model-performance reporting; external validation is still needed before deployment.


## 1. Environment setup

Activity log: define the required packages and install any that are missing from the active Jupyter environment.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "catboost": "catboost",
    "optuna": "optuna",
    "matplotlib": "matplotlib",
}

missing_packages = [
    pip_name for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", *missing_packages
    ])
else:
    print("All required packages are already installed.")

## 2. Imports and reproducibility

Activity log: import modelling, evaluation and plotting tools, then set a fixed random seed for repeatable splitting and tuning.

In [ ]:
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (    roc_auc_score, average_precision_score, accuracy_score,    precision_score, recall_score, f1_score, confusion_matrix,    classification_report, roc_curve, RocCurveDisplay, PrecisionRecallDisplay)
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Optuna:", __import__("ags").__version__)

## 3. Load the MICS child dataset

Activity log: look for the expected Stata file in the repository root. If your local filename or location differs, update `DATA_PATH` below.

In [ ]:
DATA_PATH = Path("MICS-child-ISRC.dta")
if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.dta"))
    if len(candidates) == 1:
        DATA_PATH = candidates[0]
        print(f"Using discovered dataset: {DATA_PATH}")
    elif len(candidates) == 0:
        raise FileNotFoundError("No .dta file found in the current directory. Set DATA_PATH to your MICS dataset.")
    else:
        raise FileNotFoundError(f"Multiple .dta files found: {candidates}. Set DATA_PATH explicitly.")

df = pd.read_stata(DATA_PATH, convert_categoricals=False)
print("Loaded:", DATA_PATH)
print("Dataset shape:", df.shape)


## 4. Select the requested variables and define the outcome

Retain the same 23 predictors as the severe-wasting notebook, but use `any_wasting` as the binary outcome. The notebook stops with a clear error if the target or any predictor is absent from the dataset.


In [ ]:
TARGET = "any_wasting"
PREDICTORS = [
    "age_m", "female", "division", "urban", "wb4", "parity_cat",
    "diarrhoea_2w", "cough_2w", "breathing_2w", "fuel_cat", "water_cat",
    "toilet_cat", "edu_level", "electricity", "wealth", "birth_order",
    "hh_size", "water_treated", "water_boil_filter", "handwash_place",
    "handwash_water", "floor_quality", "ari_chest"
]
REQUESTED_COLUMNS = [TARGET] + PREDICTORS
missing_columns = [col for col in REQUESTED_COLUMNS if col not in df.columns]
if missing_columns:
    raise KeyError(f"These requested variables are missing from the dataset: {missing_columns}")

model_df = df[REQUESTED_COLUMNS].copy()
print("Selected data shape:", model_df.shape)
print("Number of predictors:", len(PREDICTORS))
print("Outcome values before cleaning:", model_df[TARGET].value_counts(dropna=False).to_dict())

## 5. Complete-case cleaning and data checks

Drop rows missing the outcome or any requested predictor. Validate that `any_wasting` is binary and both classes remain. No imputation is performed.


In [ ]:
rows_before = len(model_df)
missing_summary = model_df.isna().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0].to_frame("missing_rows"))
model_df = model_df.dropna(subset=REQUESTED_COLUMNS).copy()
rows_after = len(model_df)
print(f"Rows before complete-case filtering: {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Complete cases retained: {rows_after:,} ({rows_after / rows_before:.1%})")

model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="raise").astype(int)
observed_classes = sorted(model_df[TARGET].unique().tolist())
if observed_classes != [0, 1]:
    raise ValueError(f"Expected binary outcome coded 0 and 1; found {observed_classes}. Check any_wasting coding.")

print("Complete-case outcome counts:", model_df[TARGET].value_counts().sort_index().to_dict())
print("Complete-case outcome prevalence:", model_df[TARGET].mean())


## 6. Identify categorical predictors and prepare CatBoost inputs

Activity log: declare survey-coded categories as categorical features so CatBoost does not treat their integer codes as continuous measurements. Convert categorical values to strings; keep age and household size as numeric. The list is explicit and can be adjusted if the source codebook indicates different measurement levels.

In [ ]:
CATEGORICAL_COLUMNS = [
    "female", "division", "urban", "parity_cat", "diarrhoea_2w", "cough_2w",
    "breathing_2w", "fuel_cat", "water_cat", "toilet_cat", "edu_level",
    "electricity", "wealth", "birth_order", "water_treated", "water_boil_filter",
    "handwash_place", "handwash_water", "floor_quality", "ari_chest"
]
NUMERIC_COLUMNS = [col for col in PREDICTORS if col not in CATEGORICAL_COLUMNS]

def prepare_catboost_frame(frame):
    """Return a copy with categorical columns represented consistently as strings."""
    out = frame.copy()
    for col in CATEGORICAL_COLUMNS:
        out[col] = out[col].astype(str)
    for col in NUMERIC_COLUMNS:
        out[col] = pd.to_numeric(out[col], errors="raise")
    return out

X = prepare_catboost_frame(model_df[PREDICTORS])
y = model_df[TARGET].copy()
CAT_FEATURE_INDICES = [X.columns.get_loc(col) for col in CATEGORICAL_COLUMNS]
print("Categorical predictors:", CATEGORICAL_COLUMNS)
print("Numeric predictors:", NUMERIC_COLUMNS)
print("Categorical feature indices:", CAT_FEATURE_INDICES)


## 7. Stratified train/test split

Reserve 20% of complete cases as a held-out test set and preserve the any-wasting proportion in both partitions. Optuna uses only the training partition for hyperparameter selection.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print("Training rows:", len(X_train), "| any-wasting cases:", int(y_train.sum()))
print("Test rows:", len(X_test), "| any-wasting cases:", int(y_test.sum()))


## 8. Define the CatBoost pipeline and Optuna search space

Build a scikit-learn pipeline with the clone-safe CatBoost wrapper. Optuna evaluates 30 hyperparameter configurations using stratified five-fold cross-validation and average precision (PR-AUC), a useful objective for an imbalanced outcome. CatBoost uses one CPU thread per fit.


In [ ]:
# Clone-safe wrapper prevents CatBoost estimator parameters from causing
# sklearn.clone() compatibility errors inside cross-validation workers.
class CloneSafeCatBoostClassifier(ClassifierMixin, BaseEstimator):
    def __init__(
        self, iterations=2500, depth=6, learning_rate=0.04,
        l2_leaf_reg=3.0, random_strength=1.0,
        bagging_temperature=1.0, border_count=128,
        loss_function="Logloss", eval_metric="PRAUC", random_seed=42,
        verbose=False, allow_writing_files=False, thread_count=1,
        cat_features=None,
    ):
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.l2_leaf_reg = l2_leaf_reg
        self.random_strength = random_strength
        self.bagging_temperature = bagging_temperature
        self.border_count = border_count
        self.loss_function = loss_function
        self.eval_metric = eval_metric
        self.random_seed = random_seed
        self.verbose = verbose
        self.allow_writing_files = allow_writing_files
        self.thread_count = thread_count
        self.cat_features = cat_features

    def fit(self, X, y):
        self.model_ = CatBoostClassifier(
            iterations=self.iterations, depth=self.depth,
            learning_rate=self.learning_rate, l2_leaf_reg=self.l2_leaf_reg,
            random_strength=self.random_strength,
            bagging_temperature=self.bagging_temperature,
            border_count=self.border_count, loss_function=self.loss_function,
            eval_metric=self.eval_metric, random_seed=self.random_seed,
            verbose=self.verbose, allow_writing_files=self.allow_writing_files,
            thread_count=self.thread_count,
        )
        self.model_.fit(X, y, cat_features=self.cat_features)
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    def get_feature_importance(self, *args, **kwargs):
        return self.model_.get_feature_importance(*args, **kwargs)


def make_pipeline(iterations=2500):
    model = CloneSafeCatBoostClassifier(
        iterations=iterations, random_seed=SEED, verbose=False,
        allow_writing_files=False, thread_count=1,
        cat_features=CAT_FEATURE_INDICES,
    )
    return Pipeline([
        ("prepare", FunctionTransformer(prepare_catboost_frame, validate=False)),
        ("catboost", model),
    ])

# Fixed iteration budget; tune the same CatBoost hyperparameters as the
# companion model while keeping the feature set and data split unchanged.
OPTUNA_TRIALS = 30
CV_FOLDS = 5
print("Optuna trial budget:", OPTUNA_TRIALS)
print("Cross-validation folds:", CV_FOLDS)
print("Clone-safe estimator:", type(make_pipeline().named_steps["catboost"]).__name__)


## 9. Run Optuna hyperparameter optimization (30 trials)

Optimize CatBoost with a maximum of 30 Optuna trials. Each trial is scored using stratified five-fold cross-validation average precision. The held-out test set is not used during tuning.


In [ ]:
# Confirm sklearn can clone the estimator before expensive CV begins.
base_pipeline = make_pipeline()
clone_test = clone(base_pipeline)
assert clone_test is not None
print("Estimator cloning preflight passed.")

from sklearn.model_selection import StratifiedKFold, cross_val_score

cv_strategy = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

def objective(trial):
    params = {
        "catboost__depth": trial.suggest_int("depth", 4, 10),
        "catboost__learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "catboost__l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.1, 30.0, log=True),
        "catboost__random_strength": trial.suggest_float("random_strength", 0.001, 10.0, log=True),
        "catboost__bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "catboost__border_count": trial.suggest_categorical("border_count", [64, 96, 128, 192, 254]),
    }
    candidate = clone(base_pipeline).set_params(**params)
    scores = cross_val_score(
        candidate, X_train, y_train, cv=cv_strategy,
        scoring="average_precision", n_jobs=1, error_score="raise"
    )
    trial.set_user_attr("fold_scores", scores.tolist())
    trial.set_user_attr("fold_score_std", float(np.std(scores, ddof=1)))
    return float(np.mean(scores))

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    study_name="catboost_any_wasting",
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, n_jobs=1, show_progress_bar=True)

print("Completed Optuna trials:", len(study.trials))
print("Best 5-fold average precision:", study.best_value)
print("Best CatBoost hyperparameters:")
print(json.dumps(study.best_params, indent=2))

OPTUNA_BEST_PARAMS = {
    "catboost__" + key: value for key, value in study.best_params.items()
}
best_pipeline = clone(base_pipeline).set_params(**OPTUNA_BEST_PARAMS)
best_pipeline.fit(X_train, y_train)
print("Best Optuna-tuned pipeline fitted successfully.")


## 10. Review Optuna trial history

Inspect trial parameters and cross-validation scores, then plot the best-so-far average-precision trajectory across completed trials.


In [ ]:
optuna_history_df = study.trials_dataframe(attrs=("number", "value", "params", "state", "duration"))
display(optuna_history_df.sort_values("value", ascending=False, na_position="last").head(10))

completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_numbers = [t.number + 1 for t in completed_trials]
trial_scores = [t.value for t in completed_trials]
best_so_far = np.maximum.accumulate(trial_scores)

plt.figure(figsize=(9, 5))
plt.plot(trial_numbers, trial_scores, marker="o", alpha=0.7, label="Trial CV score")
plt.plot(trial_numbers, best_so_far, linewidth=2, label="Best-so-far PR-AUC")
plt.xlabel("Optuna trial")
plt.ylabel("Mean 5-fold average precision")
plt.title("Optuna optimization history — CatBoost any wasting")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 11. Evaluate the best Optuna-tuned pipeline on the held-out test set

Fit the best Optuna configuration on the training partition and calculate discrimination and classification metrics on the untouched test partition.


In [ ]:
best_pipeline = make_pipeline()
best_pipeline.set_params(**OPTUNA_BEST_PARAMS)
best_pipeline.fit(X_train, y_train)

test_probability = best_pipeline.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

test_metrics = {
    "n_test": int(len(y_test)),
    "any_wasting_cases": int(y_test.sum()),
    "roc_auc": float(roc_auc_score(y_test, test_probability)),
    "pr_auc_average_precision": float(average_precision_score(y_test, test_probability)),
    "accuracy": float(accuracy_score(y_test, test_prediction)),
    "precision": float(precision_score(y_test, test_prediction, zero_division=0)),
    "recall": float(recall_score(y_test, test_prediction, zero_division=0)),
    "f1": float(f1_score(y_test, test_prediction, zero_division=0)),
}
display(pd.DataFrame([test_metrics]).T.rename(columns={0: "value"}))
print(classification_report(y_test, test_prediction, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, test_prediction))

RocCurveDisplay.from_predictions(y_test, test_probability)
plt.title("CatBoost any-wasting ROC curve — held-out test set")
plt.tight_layout()
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, test_probability)
plt.title("CatBoost any-wasting precision-recall curve — held-out test set")
plt.tight_layout()
plt.show()

## 12. Refit the best configuration on the overall complete-case dataset

After recording held-out evaluation, refit the selected configuration using all eligible complete cases. These full-data fitted probabilities are in-sample and must not be presented as held-out performance.


In [ ]:
final_pipeline = make_pipeline()
final_pipeline.set_params(**OPTUNA_BEST_PARAMS)
final_pipeline.fit(X, y)

model_df["predicted_probability_any_wasting"] = final_pipeline.predict_proba(X)[:, 1]
model_df["predicted_class_any_wasting"] = (
    model_df["predicted_probability_any_wasting"] >= 0.5
).astype(int)

print("Final model refitted on all complete cases:", len(model_df))
print("Observed any-wasting cases:", int(y.sum()))
print("Mean fitted probability:", model_df["predicted_probability_any_wasting"].mean())
display(model_df[[TARGET, "predicted_probability_any_wasting", "predicted_class_any_wasting"]].head())


## 13. Save model outputs and reproducibility information

Save Optuna trial history, held-out test metrics, full-data predictions, best hyperparameters and run manifest under `outputs/catboost_any_wasting_optuna/`.


In [ ]:
OUTPUT_DIR = Path("outputs") / "catboost_any_wasting_optuna"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

optuna_history_df.to_csv(OUTPUT_DIR / "optuna_trial_history.csv", index=False)
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / "heldout_test_metrics.csv", index=False)
model_df.to_csv(OUTPUT_DIR / "complete_case_predictions.csv", index=False)

with open(OUTPUT_DIR / "best_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(OPTUNA_BEST_PARAMS, f, indent=2)

manifest = {
    "data_path": str(DATA_PATH), "seed": SEED, "target": TARGET,
    "predictors": PREDICTORS, "categorical_predictors": CATEGORICAL_COLUMNS,
    "rows_before_complete_case_filter": int(rows_before),
    "complete_case_rows": int(len(model_df)), "complete_case_events": int(y.sum()),
    "optimizer": "Optuna TPE sampler", "optuna_max_trials": OPTUNA_TRIALS,
    "completed_trials": int(len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])),
    "optimization_metric": "average_precision (5-fold CV PR-AUC)",
    "best_cv_score": float(study.best_value), "best_hyperparameters": OPTUNA_BEST_PARAMS,
    "heldout_test_metrics": test_metrics,
    "note": "Full-data predictions are in-sample after refitting on all complete cases; use held-out test metrics for evaluation."
}
with open(OUTPUT_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("Saved outputs to:", OUTPUT_DIR.resolve())


## 14. Interpretation and cautions

- Optuna uses a maximum budget of 30 trials; the selected configuration is the best among those evaluated, not a guarantee of a global optimum.
- The tuning objective is mean five-fold average precision (PR-AUC), selected to reflect performance on an imbalanced outcome.
- ROC-AUC measures ranking discrimination, while precision and recall at the 0.50 threshold may be unhelpful for an uncommon outcome. Threshold selection should be performed using training/validation data if an operational threshold is needed.
- Complete-case deletion can change the analytic population and introduce selection bias. Report the number and percentage of observations removed and compare included vs excluded records where possible.
- CatBoost feature contributions and predictions are associational, not causal.
- Full-data fitted probabilities are not out-of-sample predictions. Use a separate external dataset or appropriately designed cross-validation for generalization claims.


## 15. Final evaluation summary and SHAP interpretation

Calculate precision, recall, F1-score, ROC-AUC and PR-AUC from held-out test predictions, plot the ROC curve, and calculate CatBoost-native SHAP contributions for held-out test observations. The SHAP bar chart ranks predictors by their mean absolute contribution; it does not indicate causality or direction of effect.


In [ ]:

# ============================================================
# FINAL CELL: HELD-OUT METRICS, ROC CURVE AND SHAP ANALYSIS
# ============================================================

# 1. Held-out test predictions from the best Optuna-tuned model
final_test_probability = best_pipeline.predict_proba(X_test)[:, 1]

THRESHOLD = 0.50

final_test_prediction = (
    final_test_probability >= THRESHOLD
).astype(int)


# ============================================================
# 2. CLASSIFICATION METRICS
# ============================================================

final_metrics = {
    "Precision": precision_score(
        y_test, final_test_prediction, zero_division=0
    ),
    "Recall": recall_score(
        y_test, final_test_prediction, zero_division=0
    ),
    "F1 Score": f1_score(
        y_test, final_test_prediction, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test, final_test_probability
    ),
    "PR-AUC": average_precision_score(
        y_test, final_test_probability
    ),
    "Accuracy": accuracy_score(
        y_test, final_test_prediction
    ),
}

final_metrics_df = pd.DataFrame(
    final_metrics.items(),
    columns=["Metric", "Score"]
)

print("=" * 65)
print("HELD-OUT TEST PERFORMANCE (threshold = 0.50)")
print("=" * 65)

print(
    final_metrics_df.to_string(
        index=False,
        formatters={"Score": "{:.4f}".format}
    )
)

print("\nCLASSIFICATION REPORT")
print("=" * 65)

print(classification_report(
    y_test,
    final_test_prediction,
    target_names=["Not Any Wasting", "Any Wasting"],
    zero_division=0
))

print("\nCONFUSION MATRIX")
print("=" * 65)

print(confusion_matrix(
    y_test,
    final_test_prediction
))


# ============================================================
# 3. ROC CURVE
# ============================================================

fpr, tpr, roc_thresholds = roc_curve(
    y_test,
    final_test_probability
)

roc_auc_value = roc_auc_score(
    y_test,
    final_test_probability
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"CatBoost + Optuna (AUC = {roc_auc_value:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve — Any Wasting (Held-out Test Set)")

plt.legend(loc="lower right")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# ============================================================
# 4. CATBOOST-NATIVE SHAP VALUES
#    No separate SHAP package required
# ============================================================

catboost_model = best_pipeline.named_steps["catboost"].model_

X_test_shap = prepare_catboost_frame(X_test)

shap_pool = Pool(
    data=X_test_shap,
    label=y_test,
    cat_features=CATEGORICAL_COLUMNS
)

shap_values_all = catboost_model.get_feature_importance(
    data=shap_pool,
    type="ShapValues"
)

# Last column contains the expected/base value.
# Exclude it from individual feature contributions.

shap_contributions = shap_values_all[:, :-1]


# ============================================================
# 5. GLOBAL SHAP FEATURE IMPORTANCE
# ============================================================

shap_importance_df = pd.DataFrame({
    "Variable": X_test_shap.columns,
    "Mean Absolute SHAP": np.abs(
        shap_contributions
    ).mean(axis=0)
}).sort_values(
    "Mean Absolute SHAP",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 65)
print("GLOBAL SHAP FEATURE IMPORTANCE")
print("=" * 65)

print(
    shap_importance_df.to_string(
        index=False,
        formatters={
            "Mean Absolute SHAP": "{:.6f}".format
        }
    )
)


# ============================================================
# 6. SHAP IMPORTANCE BAR GRAPH
# ============================================================

top_n = min(20, len(shap_importance_df))

bar_data = shap_importance_df.head(
    top_n
).sort_values(
    "Mean Absolute SHAP",
    ascending=True
)

plt.figure(figsize=(10, 8))

plt.barh(
    bar_data["Variable"],
    bar_data["Mean Absolute SHAP"]
)

plt.xlabel("Mean Absolute SHAP Value")
plt.ylabel("Predictor")

plt.title(
    f"CatBoost SHAP Importance — Top {top_n} Predictors"
)

plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


# ============================================================
# 7. BEESWARM-STYLE SHAP DISTRIBUTION PLOT
#    Point colour represents within-variable feature-value rank
# ============================================================

top_features = shap_importance_df.head(
    min(15, len(shap_importance_df))
)["Variable"].tolist()

rng = np.random.default_rng(SEED)

plt.figure(figsize=(11, 8))

for row, feature in enumerate(top_features):

    col_idx = X_test_shap.columns.get_loc(feature)

    shap_x = shap_contributions[:, col_idx]

    raw_values = X_test_shap[feature]

    numeric_values = pd.to_numeric(
        raw_values,
        errors="coerce"
    )

    if numeric_values.notna().sum() == 0:

        encoded_values = pd.Series(
            pd.factorize(
                raw_values.astype(str),
                sort=True
            )[0],
            index=raw_values.index,
            dtype=float
        )

    else:

        encoded_values = numeric_values.astype(float)

        if encoded_values.isna().any():
            encoded_values = encoded_values.fillna(
                encoded_values.median()
            )

    value_min = encoded_values.min()
    value_max = encoded_values.max()

    if (
        pd.isna(value_min)
        or pd.isna(value_max)
        or value_min == value_max
    ):

        colour_values = np.full(
            len(encoded_values),
            0.5
        )

    else:

        colour_values = (
            (encoded_values.to_numpy() - value_min)
            / (value_max - value_min)
        )

    jitter = rng.uniform(
        -0.28,
        0.28,
        size=len(shap_x)
    )

    plt.scatter(
        shap_x,
        row + jitter,
        c=colour_values,
        cmap="coolwarm",
        vmin=0,
        vmax=1,
        s=18,
        alpha=0.65,
        edgecolors="none"
    )

plt.axvline(
    0,
    color="black",
    linewidth=1,
    linestyle="--"
)

plt.yticks(
    range(len(top_features)),
    top_features
)

plt.gca().invert_yaxis()

plt.xlabel("SHAP value (impact on model output)")
plt.ylabel("Predictor")

plt.title(
    "SHAP Summary — Beeswarm-style Distribution (Top Predictors)"
)

colourbar = plt.colorbar()
colourbar.set_label("Relative feature value (low to high)")

plt.tight_layout()
plt.show()


# ============================================================
# 8. SAVE FINAL EVALUATION AND SHAP OUTPUTS
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save classification metrics
final_metrics_df.to_csv(
    OUTPUT_DIR / "final_evaluation_metrics.csv",
    index=False
)

# Save global SHAP feature importance
shap_importance_df.to_csv(
    OUTPUT_DIR / "shap_feature_importance.csv",
    index=False
)

# Save individual test-set SHAP contributions
pd.DataFrame(
    shap_contributions,
    columns=X_test_shap.columns
).to_csv(
    OUTPUT_DIR / "test_set_shap_values.csv",
    index=False
)


# ============================================================
# 9. COMPLETION SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FINAL EVALUATION AND SHAP ANALYSIS COMPLETED")
print("=" * 65)

print("\nGenerated files:")

print("1. final_evaluation_metrics.csv")
print("2. shap_feature_importance.csv")
print("3. test_set_shap_values.csv")

print("\nOutput directory:")
print(OUTPUT_DIR.resolve())

print("\nAll final analysis outputs have been saved successfully.")
